## Making my own GPT 

In [10]:
# Asking for Prompt From User

from datasets import load_dataset
data = load_dataset("roneneldan/TinyStories",)

In [11]:
# Tokenize the prompt
from transformers import AutoModel,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

gpt_2 = AutoModel.from_pretrained("gpt2")

tokens = tokenizer(data["train"]['text'][0],return_tensors="pt") #tokens form user prompt as per gpt-2

Vector_embeddings = gpt_2.get_input_embeddings()

embeddings = Vector_embeddings(tokens["input_ids"]) # embeddings from the gpt-2 model


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11271.12it/s]


In [12]:
# Tokens and the vector representation of it
print(f"Glimpse of tokens of the user prompt :- {tokens["input_ids"][0][0:10]}")
print(f"length tokens of the user prompt :- {len(tokens["input_ids"][0])}")

print("  ")
print("-------------------------------------")
print("  ")

print(f" shape of the Embeddings of the user prompt :- {embeddings.shape}")
print(f" glimpse of the embeddigs :- {embeddings[0][0][0:5]}")

Glimpse of tokens of the user prompt :- tensor([ 3198,  1110,    11,   257,  1310,  2576,  3706, 20037,  1043,   257])
length tokens of the user prompt :- 162
  
-------------------------------------
  
 shape of the Embeddings of the user prompt :- torch.Size([1, 162, 768])
 glimpse of the embeddigs :- tensor([-0.0672,  0.0269,  0.0795, -0.0823, -0.1724], grad_fn=<SliceBackward0>)


In [13]:
# add positional encoding to the Vector Embeddings
import torch
import numpy as np
def positional_encoding(length_of_ids,embeddings_lengths):
    # this takes the same shape as vector embeddings gives u positional embeddings for it

    pos = length_of_ids
    d = embeddings_lengths
    pe = torch.zeros((1,pos,d))
    for i in range(pos):
        for j in range(d//2):
            pe[0,i,2*j] = np.sin(i / 10000 ** (2 * j / d ))
            pe[0,i,2*j+1] = np.cos(i / 10000 ** (2 * j/ d ))
    return pe


In [14]:
#positional encodings
pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])

In [15]:
embeddings.shape

torch.Size([1, 162, 768])

In [16]:
pe.shape

torch.Size([1, 162, 768])

In [17]:
# final input embeddings = vector_embedding + positional encoding
input_embeddings = embeddings + pe

In [18]:

print(f" shape of the input embeddings of the user prompt :- {input_embeddings.shape}")
print(f" glimpse of the input embeddigs :- {embeddings[0][0][0:5]}")

 shape of the input embeddings of the user prompt :- torch.Size([1, 162, 768])
 glimpse of the input embeddigs :- tensor([-0.0672,  0.0269,  0.0795, -0.0823, -0.1724], grad_fn=<SliceBackward0>)


## Pre Attention Done

In [19]:
input_embeddings = input_embeddings.squeeze(0)

residual_1 = input_embeddings.clone()

In [20]:

#init the wieghts

w_k = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)
w_v = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)
w_q = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)
# calculating the Query , Key ,Value
Q = input_embeddings@w_q
K = input_embeddings@w_k
V = input_embeddings@w_v
Q = Q.reshape(162,12,64).transpose(1,0)
K = K.reshape(162,12,64).transpose(1,0)
V = V.reshape(162,12,64).transpose(1,0)
# Attention Score
Attention_Score = torch.nn.functional.softmax((Q@K.transpose(-2,-1))/np.sqrt(64),dim=-1)@V
Attention_score_final = Attention_Score.permute(1,0,2).reshape(input_embeddings.shape[0],768)
Attention_out = Attention_score_final + residual_1
Attention_out_rms = torch.nn.functional.rms_norm(Attention_out,(Attention_out.shape[-1],))
Attention_rms = Attention_out_rms


In [21]:
print(f"The shape of attention out {Attention_rms.shape}")
print(f"The shape of attention out {Attention_rms[0][:5]}")

The shape of attention out torch.Size([162, 768])
The shape of attention out tensor([1.0247, 1.0017, 0.9629, 1.0356, 1.0098], grad_fn=<SliceBackward0>)


## MultiHead Attention Block Done

In [22]:
residual_2 = Attention_rms.clone()
from collections import OrderedDict
d_1=Attention_rms.shape[0]
d_2=Attention_rms.shape[1]

model = torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2)),
            ("relu1", torch.nn.ReLU()),
            ("Linear2", torch.nn.Linear(d_2,d_2*4)),
            ("relu2", torch.nn.ReLU()),
            ("Linear3", torch.nn.Linear(d_2*4,d_2*4)),
            ("relu3", torch.nn.ReLU()),
            ("Linear4", torch.nn.Linear(d_2*4,d_2))
        ]
    )
)
ffn_out = model(Attention_rms)
output = ffn_out + residual_2
output = torch.nn.functional.rms_norm(output,(768,))



In [23]:

output.shape

torch.Size([162, 768])

## The FFn Layer Done


In [24]:
layer = []
for i in range(12):
    layer.append({
        "w_q" : torch.rand(768,768,requires_grad=True),
        "w_k" : torch.rand(768,768,requires_grad=True),
        "w_v" : torch.rand(768,768,requires_grad=True),
        "ffn" : torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2)),
            ("relu1", torch.nn.ReLU()),
            ("Linear2", torch.nn.Linear(d_2,d_2*4)),
            ("relu2", torch.nn.ReLU()),
            ("Linear3", torch.nn.Linear(d_2*4,d_2*4)),
            ("relu3", torch.nn.ReLU()),
            ("Linear4", torch.nn.Linear(d_2*4,d_2))
        ]
    )
)


    })


for i in range(12):
    r1 = output.clone()

    Q = output @ layer[i]["w_q"]
    K = output @ layer[i]["w_k"]
    V = output @ layer[i]["w_v"]

    Q = Q.reshape(output.shape[0],12,64).transpose(1,0)
    K = K.reshape(output.shape[0],12,64).transpose(1,0)
    V = V.reshape(output.shape[0],12,64).transpose(1,0)


    Attention_score = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64) ,dim=-1)@V
    Attention_score = Attention_score.permute(1,0,2).reshape(output.shape[0],768)
    Attn_out = Attention_score  + r1
    Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

    r2 = Attn_rms.clone()

    ffn_out = layer[i]["ffn"](Attn_rms)

    ffn_out = ffn_out + r2

    ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
    output = ffn_out





all_params = []

for i in layer:
    all_params += [i["w_k"],i["w_v"],i["w_q"]]
    all_params += list(i["ffn"].parameters())

all_params += list(model.parameters())
all_params += [w_k,w_q,w_v]




In [25]:
vocab = 50257

output_projection = torch.nn.Linear(768,vocab)

logits = output_projection(output)

all_params += list(output_projection.parameters())

ids = tokens["input_ids"].squeeze(0)
target = torch.cat([ids[1:],ids[-1:]],dim=0)

optimizer = torch.optim.Adam(all_params,lr=1e-2)
loss = torch.nn.functional.cross_entropy(logits,target)
optimizer = torch.optim.Adam(all_params,lr=1e-2)



In [26]:
losses = []
for j in range(100):
    embeddings = Vector_embeddings(tokens["input_ids"])
    pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])
    input_embeddings = embeddings + pe
    input_embeddings = input_embeddings.squeeze(0)
    output = input_embeddings

    residual_1 = input_embeddings.clone()
    Q = input_embeddings @ w_q
    K = input_embeddings @ w_k
    V = input_embeddings @ w_v
    seq_len = input_embeddings.shape[0]
    Q = Q.reshape(seq_len, 12, 64).transpose(1, 0)
    K = K.reshape(seq_len, 12, 64).transpose(1, 0)
    V = V.reshape(seq_len, 12, 64).transpose(1, 0)
    attn = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64), dim=-1) @ V
    attn = attn.permute(1, 0, 2).reshape(seq_len, 768)
    output = torch.nn.functional.rms_norm(attn + residual_1, (768,))

    # train
    for i in range(12):

        r1 = output.clone()

        Q = output @ layer[i]["w_q"]
        K = output @ layer[i]["w_k"]
        V = output @ layer[i]["w_v"]

        Q = Q.reshape(output.shape[0],12,64).transpose(1,0)
        K = K.reshape(output.shape[0],12,64).transpose(1,0)
        V = V.reshape(output.shape[0],12,64).transpose(1,0)


        Attention_score = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64) ,dim=-1)@V
        Attention_score = Attention_score.permute(1,0,2).reshape(output.shape[0],768)
        Attn_out = Attention_score  + r1
        Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

        r2 = Attn_rms.clone()

        ffn_out = layer[i]["ffn"](Attn_rms)

        ffn_out = ffn_out + r2

        ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
        output = ffn_out

    # calc loss backprop and update weights
    logits = output_projection(output)
    loss = torch.nn.functional.cross_entropy(logits,target)

    optimizer.zero_grad()
    torch.autograd.set_detect_anomaly(True, check_nan=False)
    loss.backward(retain_graph=True,)
    optimizer.step()

    losses.append(loss)

    if j%10 == 0:
        print(f"The loss at iteration {j} is {loss}")




The loss at iteration 0 is 11.074292182922363
The loss at iteration 10 is 5.307166576385498
The loss at iteration 20 is 4.8559064865112305
The loss at iteration 30 is 4.412309646606445
The loss at iteration 40 is 4.192607402801514
The loss at iteration 50 is 4.152687072753906
The loss at iteration 60 is 4.130831241607666
The loss at iteration 70 is 4.118578910827637
The loss at iteration 80 is 4.115963935852051
The loss at iteration 90 is 4.115070343017578


In [27]:
output.shape

torch.Size([162, 768])

In [28]:
def generate(prompt,max_tokens = 20,temp = 1.0):
    tokens = tokenizer(prompt,return_tensors="pt")
    ids = tokens["input_ids"].squeeze(0)

    for _ in range(max_tokens):
        embeddings = Vector_embeddings(ids)
        pe = positional_encoding(embeddings.shape[0],embeddings.shape[1])
        input_embeddings = embeddings + pe
        input_embeddings = input_embeddings.squeeze(0)
        output = input_embeddings

        residual_1 = input_embeddings.clone()
        Q = input_embeddings @ w_q
        K = input_embeddings @ w_k
        V = input_embeddings @ w_v
        seq_len = input_embeddings.shape[0]
        Q = Q.reshape(seq_len, 12, 64).transpose(1, 0)
        K = K.reshape(seq_len, 12, 64).transpose(1, 0)
        V = V.reshape(seq_len, 12, 64).transpose(1, 0)
        attn = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64), dim=-1) @ V
        attn = attn.permute(1, 0, 2).reshape(seq_len, 768)
        output = torch.nn.functional.rms_norm(attn + residual_1, (768,))

        # train
        for i in range(12):

            r1 = output.clone()

            Q = output @ layer[i]["w_q"]
            K = output @ layer[i]["w_k"]
            V = output @ layer[i]["w_v"]

            Q = Q.reshape(output.shape[0],12,64).transpose(1,0)
            K = K.reshape(output.shape[0],12,64).transpose(1,0)
            V = V.reshape(output.shape[0],12,64).transpose(1,0)


            Attention_score = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64) ,dim=-1)@V
            Attention_score = Attention_score.permute(1,0,2).reshape(output.shape[0],768)
            Attn_out = Attention_score  + r1
            Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

            r2 = Attn_rms.clone()

            ffn_out = layer[i]["ffn"](Attn_rms)

            ffn_out = ffn_out + r2

            ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
            output = ffn_out



            logits = output_projection(output[-1])

            logits = logits/temp

            probs = torch.nn.functional.softmax(logits,dim=-1)

            next_token = torch.multinomial(probs,num_samples=1)

            ids = torch.cat([ids,next_token],dim=0)

            if next_token == tokenizer.eos_token_id:
                break


    return tokenizer.decode(ids,skip_special_tokens=True)



In [29]:
prompt = input("Enter the prompt : - ")

ans = generate(prompt,max_tokens=20,temp=1.5)

In [30]:
ans

'lilly.....,.. suppressed. (); on,.... mom,. bureaucrats. Hawking little.......,hide finishedACK them... knew.,.,monary room vagina,,,...... optim button soft they..L.. sew../// together Minecraft wanted........icity Herforcing SheMom said. day....ispers. coincides not.. each It. Her..velt each spoken....,..Mom.508.HE se share.. this. and,. deserve. revered difficult happy....... whatsoever needleyden\n,.,.....883 wanted biting button,... in,,, fascinated.external in... difficult,...zing. sects,. needle., Her... penal sharp Filip on..,. this... debunk knew proto shirt,,...,.. Investments I autistic sew... they.... dig.wide had room....,.. miss, roll and'